In [1]:
import scanpy as sc
import omicverse as ov
import pandas as pd
ov.plot_set()


   ____            _     _    __                  
  / __ \____ ___  (_)___| |  / /__  _____________ 
 / / / / __ `__ \/ / ___/ | / / _ \/ ___/ ___/ _ \ 
/ /_/ / / / / / / / /__ | |/ /  __/ /  (__  )  __/ 
\____/_/ /_/ /_/_/\___/ |___/\___/_/  /____/\___/                                              

Version: 1.6.11, Tutorials: https://omicverse.readthedocs.io/
Dependency error: The 'phate>=1.0' distribution was not found and is required by the application


In [2]:
adata = sc.read("/home/lugli/spuccio/Projects/SP039/GBmap/Ruiz2022_Part2.h5ad")

In [3]:
adata = adata[adata.obs['donor_id'].isin(["NH16-2366", "NH17-1245", "NH17-1258", "NH17-161", "NH17-1953", "NH17-2069", "NH17-2680", "NH17-442", "NH18-1406", "NH19-123", "NH19-565"])]

In [4]:
df_obs = pd.DataFrame(adata.obs)

In [5]:
del adata.obs

In [6]:
adata = adata.raw.to_adata()

In [7]:
adata

AnnData object with n_obs × n_vars = 39124 × 18189
    var: 'mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances', 'residual_variances', 'highly_variable_rank', 'highly_variable_features'
    uns: 'X_approximate_distribution', 'annotation_level_1_colors', 'annotation_level_2_colors', 'annotation_level_3_colors', 'batch_condition', 'default_embedding', 'donor_id_colors', 'hvg', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'rank_genes_groups', 'scaled|original|cum_sum_eigenvalues', 'scaled|original|pca_var_ratios', 'schema_version', 'scsa_celltype_cellmarker_colors', 'scsa_celltype_panglaodb_colors', 'title', 'umap'
    obsm: 'X_harmony', 'X_pca', 'X_umap', 'scaled|original|X_pca'
    obsp: 'connectivities', 'distances'

In [9]:
#adata = adata.raw.to_adata()

In [10]:
X_counts_recovered, size_factors_sub=ov.pp.recover_counts(adata.X, 50*1e4, 50*1e5, log_base=None, 
                                                          chunk_size=10000)


100%|██████████| 9124/9124 [00:12<00:00, 723.42it/s]


In [11]:
adata.X = X_counts_recovered

In [12]:
annot = sc.queries.biomart_annotations(
    "hsapiens",
    ["external_gene_name","ensembl_gene_id", "start_position", "end_position", "chromosome_name",],
).set_index("external_gene_name")

In [13]:
annot

,ensembl_gene_id,start_position,end_position,chromosome_name
external_gene_name,,,,
MT-TF,ENSG00000210049,577,647,MT
MT-RNR1,ENSG00000211459,648,1601,MT
MT-TV,ENSG00000210077,1602,1670,MT
MT-RNR2,ENSG00000210082,1671,3229,MT
MT-TL1,ENSG00000209082,3230,3304,MT
...,...,...,...,...
SCMH1-DT,ENSG00000235358,41241772,41338644,1
LINC01740,ENSG00000228067,212467563,212556085,1
SLC44A3-AS1,ENSG00000293271,94585556,94855426,1


In [14]:
adata.var.columns

Index(['mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances',
       'residual_variances', 'highly_variable_rank',
       'highly_variable_features'],
      dtype='object')

In [15]:
adata.var = adata.var[['mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances',
       'residual_variances']]

In [16]:
adata.var 

,mt,n_cells,percent_cells,robust,means,variances,residual_variances
feature_name,,,,,,,
ZNF367,False,3013,7.701155,True,0.048643,0.036374,0.748558
SULT1B1,False,248,0.633882,True,0.004324,0.003990,0.998424
TRIM63,False,372,0.950823,True,0.005832,0.004591,0.805799
HDHD2,False,788,2.014109,True,0.009969,0.006378,0.683130
MORF4L2-AS1,False,797,2.037113,True,0.011250,0.007690,0.719983
...,...,...,...,...,...,...,...
LINC02498,False,361,0.922707,True,0.005141,0.003443,0.659887
LINC02105,False,123,0.314385,True,0.001533,0.000920,0.610718
LINC02546,False,106,0.270933,True,0.001379,0.000893,0.658389


In [17]:
df_tmp = pd.merge(adata.var , annot, left_index=True, right_index=True, how='left')

In [18]:
df_tmp = df_tmp.reset_index().drop_duplicates(['feature_name']).set_index(['feature_name'])

In [19]:
adata.var = df_tmp

In [20]:
adata = adata[:,adata.var['chromosome_name'].isin(["1","2","3","4","5","6","7","8","9","10","11","12","13","14","15","16","17","18","19","20","21","22","X","Y","MT"])]

In [21]:
adata

View of AnnData object with n_obs × n_vars = 39124 × 16810
    var: 'mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances', 'residual_variances', 'ensembl_gene_id', 'start_position', 'end_position', 'chromosome_name'
    uns: 'X_approximate_distribution', 'annotation_level_1_colors', 'annotation_level_2_colors', 'annotation_level_3_colors', 'batch_condition', 'default_embedding', 'donor_id_colors', 'hvg', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'rank_genes_groups', 'scaled|original|cum_sum_eigenvalues', 'scaled|original|pca_var_ratios', 'schema_version', 'scsa_celltype_cellmarker_colors', 'scsa_celltype_panglaodb_colors', 'title', 'umap'
    obsm: 'X_harmony', 'X_pca', 'X_umap', 'scaled|original|X_pca'
    obsp: 'connectivities', 'distances'

In [22]:
adata.obs['donor_id'] = df_obs['donor_id']

In [25]:
metadata_data = {
    'Author': ['Ruiz2022'] * 11,
    'donor_id': ["NH16-2366", "NH17-1245", "NH17-1258", "NH17-161", "NH17-1953", "NH17-2069", 
                 "NH17-2680", "NH17-442", "NH18-1406", "NH19-123", "NH19-565"],
    'stage': ['Primary'] * 11,
    'assay': ['10x 3\' v3'] * 11,
    'tissue': ["left frontal lobe", "left temporal lobe", "right temporal lobe", "brain", "right temporal lobe", 
               "brain", "forebrain", "brain", "right frontal lobe", "brain", "right temporal lobe"],
    'Cells': ['Total'] * 11,
    'Method': ['cell'] * 11
}

metadata_df = pd.DataFrame(metadata_data)

# Display the metadata DataFrame
print(metadata_df)

      Author   donor_id    stage      assay               tissue  Cells Method
0   Ruiz2022  NH16-2366  Primary  10x 3' v3    left frontal lobe  Total   cell
1   Ruiz2022  NH17-1245  Primary  10x 3' v3   left temporal lobe  Total   cell
2   Ruiz2022  NH17-1258  Primary  10x 3' v3  right temporal lobe  Total   cell
3   Ruiz2022   NH17-161  Primary  10x 3' v3                brain  Total   cell
4   Ruiz2022  NH17-1953  Primary  10x 3' v3  right temporal lobe  Total   cell
5   Ruiz2022  NH17-2069  Primary  10x 3' v3                brain  Total   cell
6   Ruiz2022  NH17-2680  Primary  10x 3' v3            forebrain  Total   cell
7   Ruiz2022   NH17-442  Primary  10x 3' v3                brain  Total   cell
8   Ruiz2022  NH18-1406  Primary  10x 3' v3   right frontal lobe  Total   cell
9   Ruiz2022   NH19-123  Primary  10x 3' v3                brain  Total   cell
10  Ruiz2022   NH19-565  Primary  10x 3' v3  right temporal lobe  Total   cell


In [26]:
merged_obs_df = pd.merge(pd.DataFrame(adata.obs), metadata_df, left_on='donor_id', right_on='donor_id', how='left')

# Display the merged dataframe
print(merged_obs_df)

        donor_id    Author    stage      assay               tissue  Cells  \
0      NH16-2366  Ruiz2022  Primary  10x 3' v3    left frontal lobe  Total   
1      NH16-2366  Ruiz2022  Primary  10x 3' v3    left frontal lobe  Total   
2      NH16-2366  Ruiz2022  Primary  10x 3' v3    left frontal lobe  Total   
3      NH16-2366  Ruiz2022  Primary  10x 3' v3    left frontal lobe  Total   
4      NH16-2366  Ruiz2022  Primary  10x 3' v3    left frontal lobe  Total   
...          ...       ...      ...        ...                  ...    ...   
39119   NH19-565  Ruiz2022  Primary  10x 3' v3  right temporal lobe  Total   
39120   NH19-565  Ruiz2022  Primary  10x 3' v3  right temporal lobe  Total   
39121   NH19-565  Ruiz2022  Primary  10x 3' v3  right temporal lobe  Total   
39122   NH19-565  Ruiz2022  Primary  10x 3' v3  right temporal lobe  Total   
39123   NH19-565  Ruiz2022  Primary  10x 3' v3  right temporal lobe  Total   

      Method  
0       cell  
1       cell  
2       cell  
3  

In [27]:
df_obs = df_obs[['donor_id','n_genes','nUMIs','annotation_level_1', 'annotation_level_2','annotation_level_3','scsa_celltype_cellmarker', 'scsa_celltype_panglaodb','cell_type']]

In [29]:
df_obs

,donor_id,n_genes,nUMIs,annotation_level_1,annotation_level_2,annotation_level_3,scsa_celltype_cellmarker,scsa_celltype_panglaodb,cell_type
NH16-2366_AAACCCAAGATCGCCC-1-1-1,NH16-2366,1199,1738.116211,Non-neoplastic,Myeloid,TAM-MG,Microglial cell,Macrophages,microglial cell
NH16-2366_AAACCCACACACTTAG-1-1-1,NH16-2366,1278,1754.206787,Non-neoplastic,Lymphoid,CD4/CD8,Microglial cell,T Cells,mature T cell
NH16-2366_AAACCCATCTTCCTAA-1-1-1,NH16-2366,2847,2172.254395,Non-neoplastic,Glial-Neuronal,Oligodendrocyte,Oligodendrocyte,Oligodendrocytes,oligodendrocyte
NH16-2366_AAAGAACCACATGACT-1-1-1,NH16-2366,3156,2145.464111,Non-neoplastic,Myeloid,TAM-BDM,Microglial cell,Macrophages,macrophage
NH16-2366_AAAGAACTCAAGCTGT-1-1-1,NH16-2366,4782,2332.191650,Non-neoplastic,Glial-Neuronal,Oligodendrocyte,Oligodendrocyte,Oligodendrocytes,oligodendrocyte
...,...,...,...,...,...,...,...,...,...
NH19-565_TTTGTTGGTACTGAGG-1-1-1,NH19-565,2309,2061.813477,Non-neoplastic,Glial-Neuronal,Oligodendrocyte,Oligodendrocyte,Oligodendrocytes,oligodendrocyte
NH19-565_TTTGTTGGTTCCTTGC-1-1-1,NH19-565,3290,2145.987793,Neoplastic,Stem-like,OPC-like,Astrocyte,Fibroblasts,malignant cell
NH19-565_TTTGTTGTCAAAGGTA-1-1-1,NH19-565,2709,2165.189941,Non-neoplastic,Glial-Neuronal,Oligodendrocyte,Oligodendrocyte,Oligodendrocytes,oligodendrocyte
NH19-565_TTTGTTGTCAATGTCG-1-1-1,NH19-565,2680,2157.970459,Non-neoplastic,Glial-Neuronal,Oligodendrocyte,Oligodendrocyte,Oligodendrocytes,oligodendrocyte


In [30]:
merged_obs_df.index= df_obs.index

In [31]:
merged_obs_df.columns

Index(['donor_id', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method'], dtype='object')

In [32]:
merged_obs_df = pd.merge(merged_obs_df, df_obs,right_index=True,left_index=True, how='left')

In [33]:
merged_obs_df.columns

Index(['donor_id_x', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method',
       'donor_id_y', 'n_genes', 'nUMIs', 'annotation_level_1',
       'annotation_level_2', 'annotation_level_3', 'scsa_celltype_cellmarker',
       'scsa_celltype_panglaodb', 'cell_type'],
      dtype='object')

In [34]:
del merged_obs_df['donor_id_y']

In [35]:
merged_obs_df.columns = ['donor_id', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method',
                         'n_genes', 'nUMIs', 'annotation_level_1',
       'annotation_level_2', 'annotation_level_3', 'scsa_celltype_cellmarker',
       'scsa_celltype_panglaodb', 'cell_type']

In [36]:
merged_obs_df.columns

Index(['donor_id', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method',
       'n_genes', 'nUMIs', 'annotation_level_1', 'annotation_level_2',
       'annotation_level_3', 'scsa_celltype_cellmarker',
       'scsa_celltype_panglaodb', 'cell_type'],
      dtype='object')

In [37]:
adata.obs = merged_obs_df

In [38]:
ov.pp.score_genes_cell_cycle(adata,species='human')

calculating cell cycle phase
computing score 'S_score'
    finished: added
    'S_score', score of gene set (adata.obs).
    598 total control genes are used. (0:00:01)
computing score 'G2M_score'
    finished: added
    'G2M_score', score of gene set (adata.obs).
    812 total control genes are used. (0:00:01)
-->     'phase', cell cycle phase (adata.obs)


In [39]:
adata.write("/home/lugli/spuccio/Projects/SP039/GBmap/Ruiz2022_Part3.h5ad")